# 04. Model Comparison
WElo vs Logistic Regression vs XGBoost with TimeSeriesSplit.

In [ ]:
import sys
sys.path.insert(0, '..')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import roc_curve, auc, accuracy_score, log_loss, brier_score_loss, roc_auc_score

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('imports ok')

In [ ]:
fm = pd.read_parquet('../data/processed/feature_matrix.parquet')
print(f'Feature matrix: {fm.shape}')

delta_cols = [c for c in fm.columns if c.endswith('_delta')]
h2h_cols = [c for c in fm.columns if c.startswith('h2h_')]
feature_cols = delta_cols + h2h_cols + ['surface_grass']
print(f'Feature cols: {len(feature_cols)}')

TEST_YEAR_CUTOFF = 2025
RANDOM_STATE = 42

train = fm[fm['tourney_date'].dt.year < TEST_YEAR_CUTOFF].copy()
test = fm[fm['tourney_date'].dt.year >= TEST_YEAR_CUTOFF].copy()
print(f'Train: {len(train):,} rows | Test: {len(test):,} rows')

In [ ]:
def flip_rows(df: pd.DataFrame, feature_cols: list, seed: int) -> pd.DataFrame:
    """Randomly flip half the rows to ensure model symmetry."""
    rng = np.random.default_rng(seed)
    mask = rng.random(len(df)) < 0.5
    flipped = df.copy()
    for col in feature_cols:
        if col.startswith('h2h_') or col == 'surface_grass':
            continue
        flipped.loc[mask, col] = -flipped.loc[mask, col]
    if 'h2h_win_pct_w' in flipped.columns:
        flipped.loc[mask, 'h2h_win_pct_w'] = 1 - flipped.loc[mask, 'h2h_win_pct_w']
    if 'h2h_grass_win_pct_w' in flipped.columns:
        flipped.loc[mask, 'h2h_grass_win_pct_w'] = 1 - flipped.loc[mask, 'h2h_grass_win_pct_w']
    if 'h2h_recent_3_w' in flipped.columns:
        flipped.loc[mask, 'h2h_recent_3_w'] = 1 - flipped.loc[mask, 'h2h_recent_3_w']
    flipped.loc[mask, 'y'] = 0
    return flipped

test_flipped = flip_rows(test, feature_cols, RANDOM_STATE + 1)
X_test = test_flipped[feature_cols].fillna(0)
y_test = test_flipped['y'].values
print(f'Test set after flip: {len(X_test):,} rows, y=1 rate: {y_test.mean():.2%}')

## Weighted Elo baseline

In [ ]:
from src.models.elo_model import WeightedEloPredictor

elo_model = WeightedEloPredictor()
y_proba_elo = elo_model.predict_proba(X_test)[:, 1]
y_pred_elo = (y_proba_elo >= 0.5).astype(int)

elo_metrics = {
    'model': 'Weighted Elo',
    'accuracy': accuracy_score(y_test, y_pred_elo),
    'log_loss': log_loss(y_test, y_proba_elo),
    'brier_score': brier_score_loss(y_test, y_proba_elo),
    'roc_auc': roc_auc_score(y_test, y_proba_elo),
}
print(elo_metrics)

## Logistic Regression

In [ ]:
lr_model = joblib.load('../models/logistic_pipeline.pkl')
y_proba_lr = lr_model.predict_proba(X_test)[:, 1]
y_pred_lr = lr_model.predict(X_test)

lr_metrics = {
    'model': 'Logistic Regression',
    'accuracy': accuracy_score(y_test, y_pred_lr),
    'log_loss': log_loss(y_test, y_proba_lr),
    'brier_score': brier_score_loss(y_test, y_proba_lr),
    'roc_auc': roc_auc_score(y_test, y_proba_lr),
}
print(lr_metrics)

## XGBoost

In [ ]:
xgb_model = joblib.load('../models/xgb_pipeline.pkl')
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = xgb_model.predict(X_test)

xgb_metrics = {
    'model': 'XGBoost',
    'accuracy': accuracy_score(y_test, y_pred_xgb),
    'log_loss': log_loss(y_test, y_proba_xgb),
    'brier_score': brier_score_loss(y_test, y_proba_xgb),
    'roc_auc': roc_auc_score(y_test, y_proba_xgb),
}
print(xgb_metrics)

## Comparison table

In [ ]:
results = pd.DataFrame([elo_metrics, lr_metrics, xgb_metrics])
results = results.set_index('model')
results.columns = ['Accuracy', 'Log Loss', 'Brier Score', 'ROC AUC']
results.style.format('{:.4f}').background_gradient(cmap='RdYlGn', subset=['Accuracy', 'ROC AUC']).background_gradient(cmap='RdYlGn_r', subset=['Log Loss', 'Brier Score'])

## ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

model_probs = [
    ('Weighted Elo', y_proba_elo, 'steelblue'),
    ('Logistic Regression', y_proba_lr, 'darkorange'),
    ('XGBoost', y_proba_xgb, 'seagreen'),
]

for name, proba, color in model_probs:
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.4f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models (Test Set 2025+)')
ax.legend(loc='lower right')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Feature importance (XGBoost)

In [ ]:
# XGBoost pipeline: last step is the booster
import xgboost as xgb

# Extract XGB step from pipeline
xgb_step = xgb_model  # may be pipeline or model directly
if hasattr(xgb_step, 'named_steps'):
    # It's a sklearn Pipeline
    for step_name, step in xgb_step.named_steps.items():
        if hasattr(step, 'feature_importances_'):
            xgb_step = step
            break

importances = pd.Series(
    xgb_step.feature_importances_,
    index=feature_cols
).nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
importances.plot.barh(ax=ax, color='seagreen')
ax.set_xlabel('Feature importance (gain)')
ax.set_title('XGBoost — Top 15 Feature Importances')
plt.tight_layout()
plt.show()